# Этап 3 — обучение и проверка сохранённых SB3 PPO моделей
**Новая интеграция audit_only подготовлена, но ещё не выполнялась в Colab.**
По умолчанию включён `audit_only`: существующие модели загружаются без обучения.
Smoke commit a768abc подтверждён пользовательскими артефактами: 97 тестов, по 4096 шагов
на режим. Между runtime обнаружено небольшое расхождение действий; точное совпадение не заявляется.

`GIT_REF` — версия кода **проверки**, `MODEL_GIT_SHA` — версия кода **обучения** старых моделей.
Они могут различаться и сохраняются отдельно. Старые результаты не перезаписываются.
Только train/validation; test не загружается. Читайте `docs/STAGE3_VALIDATION.md`.

## 1. GitHub и параметры запуска — редактировать здесь
Опубликуйте новый commit с этой версией notebook и откройте notebook из него.
В `GIT_REF` задайте его полный SHA (при первом запуске допустим `main`).
Для текущих сохранённых моделей оставьте `MODEL_GIT_SHA` и три `RELOAD_RUN_IDS` как ниже,
`EXECUTION_MODE="audit_only"`, `DEVICE="cpu"`. Выполняйте сверху вниз в новой сессии Colab.

Режимы: `audit_only` — диагностический аудит без обучения и автоматических допусков;
`reload_only` — строгий exact replay, который завершится ошибкой при отличии от исходного CSV;
`train` — явное новое обучение с конфигурацией smoke/pilot. Изменение режима не выполняется автоматически.

In [ ]:
import sys
import os
import re
import json
import subprocess
import tempfile
from pathlib import Path

try:
    import google.colab
except ImportError as error:
    raise RuntimeError("Запуск разрешён только в Google Colab, не в локальном runtime") from error
if not Path("/content").is_dir() or "COLAB_RELEASE_TAG" not in os.environ:
    raise RuntimeError("Нужна управляемая сессия Google Colab с каталогом /content")
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Нужен Python 3.11/3.12. Выберите Runtime > Change runtime type > Runtime Version 2026.07 (Python 3.12).")

GITHUB_REPOSITORY = "Go9shi/Science_Paper"
GIT_REF = "main"  # Evaluator commit; use its full SHA for reproducibility.
USE_GITHUB_SECRET = False
CHECK_RL_COMPATIBILITY = True
RUN_KIND = "smoke"  # "smoke" (4096/mode) or "pilot" (65536/mode); no parameter search.
EXECUTION_MODE = "audit_only"  # "audit_only", "reload_only", or explicit "train".
MODEL_GIT_SHA = "a768abcadd295f492d35c6f023b5efcad86fb9cb"  # Original TRAINING commit.
DEVICE = "cpu"  # Optional: "cuda". No silent fallback.
REWARD_MODES = ("none", "fixed", "adaptive")
RELOAD_RUN_IDS = {
    "none": "20261107T190619Z-none-ad65b98b",
    "fixed": "20261107T190633Z-fixed-d87313a4",
    "adaptive": "20261107T190645Z-adaptive-6f734f42",
}
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
if RUN_KIND not in {"smoke", "pilot"} or EXECUTION_MODE not in {"train", "reload_only", "audit_only"}:
    raise ValueError("Unknown run kind/execution mode")
if DEVICE not in {"cpu", "cuda"} or REWARD_MODES != ("none", "fixed", "adaptive"):
    raise ValueError("Use a common cpu/cuda device and all three matched modes")
if EXECUTION_MODE != "train" and set(RELOAD_RUN_IDS) != set(REWARD_MODES):
    raise ValueError("Fill RELOAD_RUN_IDS for all three saved-model modes")
if any(not re.fullmatch(r"[A-Za-z0-9_-]+", value) for value in RELOAD_RUN_IDS.values()):
    raise ValueError("Invalid reload run ID")
if EXECUTION_MODE != "train" and not re.fullmatch(r"[0-9a-fA-F]{40}", MODEL_GIT_SHA):
    raise ValueError("MODEL_GIT_SHA must be the full original training commit SHA")
SOURCE_RUN_ID = "20261107T154124Z-9f1f7c1e"  # Existing complete Stage 1 run; no download.
NOTEBOOK_REVISION = "ppo-infrastructure-v2-audit"
REPO_URL = f"https://github.com/{GITHUB_REPOSITORY}.git"
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", GITHUB_REPOSITORY):
    raise ValueError("Ожидается repository в формате owner/name")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]*", GIT_REF):
    raise ValueError("Некорректный GIT_REF")
if not re.fullmatch(r"[A-Za-z0-9_-]+", SOURCE_RUN_ID):
    raise ValueError("Некорректный SOURCE_RUN_ID")

REPO_ROOT = Path(tempfile.mkdtemp(prefix="Science_Paper-stage3-", dir="/content"))
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
with tempfile.TemporaryDirectory(prefix="stage1-auth-", dir="/content") as auth_dir:
    if USE_GITHUB_SECRET:
        from google.colab import userdata
        try:
            git_env["STAGE1_GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
        except Exception as error:
            raise RuntimeError("Разрешите доступ к Colab Secret GITHUB_TOKEN с правом чтения репозитория") from error
        if not git_env["STAGE1_GITHUB_TOKEN"]:
            raise RuntimeError("GITHUB_TOKEN пуст")
        askpass = Path(auth_dir) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            "print('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['STAGE1_GITHUB_TOKEN'])\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        git_env["GIT_ASKPASS"] = str(askpass)
    try:
        subprocess.run(["git", "-c", "credential.helper=", "clone", "--quiet", "--no-checkout",
                        REPO_URL, str(REPO_ROOT)], env=git_env, check=True, capture_output=True, text=True, timeout=180)
        if re.fullmatch(r"[0-9a-fA-F]{7,40}", GIT_REF):
            # Full clone already contains reachable history; resolve short SHAs locally.
            resolved = subprocess.run(
                ["git", "-C", str(REPO_ROOT), "rev-parse", "--verify", f"{GIT_REF}^{{commit}}"],
                capture_output=True, text=True,
            )
            if resolved.returncode == 0:
                SOURCE_COMMIT = resolved.stdout.strip()
            elif len(GIT_REF) < 40:
                raise ValueError("Short SHA was not found or is ambiguous. Use the full 40-character commit SHA.")
            else:
                subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                               env=git_env, check=True, capture_output=True, text=True, timeout=180)
                SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        else:
            subprocess.run(["git", "-c", "credential.helper=", "-C", str(REPO_ROOT), "fetch", "--quiet", "origin", GIT_REF],
                           env=git_env, check=True, capture_output=True, text=True, timeout=180)
            SOURCE_COMMIT = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "FETCH_HEAD"], text=True).strip()
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
        raise RuntimeError("Не удалось получить код GitHub. Проверьте сеть, GIT_REF и доступ; для private используйте Secret.") from error
    finally:
        git_env.pop("STAGE1_GITHUB_TOKEN", None)
        git_env.pop("GIT_ASKPASS", None)

PROJECT_ROOT = REPO_ROOT / "research"
required = ["AGENTS.md", "src/__init__.py", "src/data.py", "src/volatility.py", "src/validation.py", "src/environment.py",
            "tests/test_data.py", "tests/test_volatility.py", "tests/test_validation.py", "tests/test_environment.py",
            "requirements-colab.txt", "requirements-rl-check.txt", "docs/METHODOLOGY.md", "docs/STAGE1_VALIDATION.md", "docs/COLAB_ENVIRONMENT.md",
            "notebooks/01_data_preparation.ipynb"]
if not CHECK_RL_COMPATIBILITY:
    raise RuntimeError("Stage 3 requires CHECK_RL_COMPATIBILITY=True")
required += ["src/portfolio.py", "src/environment_data.py", "src/portfolio_env.py", "src/env_validation.py",
             "tests/test_portfolio_env.py", "docs/STAGE2_VALIDATION.md", "notebooks/02_environment_validation.ipynb"]
required += ["src/training_config.py", "src/training_data.py", "src/ppo_training.py", "src/financial_metrics.py",
             "src/regularization.py", "src/replay_audit.py", "configs/ppo_smoke.json", "configs/ppo_pilot.json",
             "tests/test_ppo_training.py", "docs/STAGE3_VALIDATION.md", "notebooks/03_ppo_training.ipynb"]
missing = [name for name in required if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Commit {SOURCE_COMMIT} не содержит актуальный этап 1: {missing}. "
                            "Опубликуйте файлы и выберите содержащий их ref; локальные файлы не подменяют GitHub.")
source_notebook = json.loads((PROJECT_ROOT / "notebooks/03_ppo_training.ipynb").read_text(encoding="utf-8"))
if source_notebook["metadata"].get("stage3_revision") != NOTEBOOK_REVISION:
    raise RuntimeError("Версия открытого notebook не соответствует checkout; откройте notebook из выбранного commit")
print("GitHub:", GITHUB_REPOSITORY)
print("Resolved commit:", SOURCE_COMMIT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. Google Drive: проверка чтения и записи

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("Google Drive не подключён: каталог MyDrive недоступен")
OUTPUT_ROOT = DRIVE_ROOT / "ppo_turnover_research" / "stage3"
PREFLIGHT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
PREFLIGHT_DIR = OUTPUT_ROOT / "preflight" / PREFLIGHT_ID
try:
    PREFLIGHT_DIR.mkdir(parents=True, exist_ok=False)
    probe = PREFLIGHT_DIR / "drive-probe.txt"
    probe_value = uuid4().hex
    probe.write_text(probe_value, encoding="utf-8")
    if probe.read_text(encoding="utf-8") != probe_value:
        raise OSError("Drive read-back mismatch")
    probe.unlink()
except OSError as error:
    raise RuntimeError(f"Нет надёжного чтения/записи Google Drive: {OUTPUT_ROOT}") from error
print("Drive read/write: OK")
print("Preflight logs:", PREFLIGHT_DIR)
STAGE1_RUN = DRIVE_ROOT / "ppo_turnover_research" / "stage1" / "runs" / SOURCE_RUN_ID
if not (STAGE1_RUN / "metadata.json").is_file():
    raise FileNotFoundError(f"Complete Stage 1 directory is unavailable: {STAGE1_RUN}")
print("Stage 1 source:", STAGE1_RUN)

## 3. Зафиксированные зависимости и preflight
Профиль env-v2: Colab 2026.07 / Python 3.12, NumPy 2.0.2, pandas 2.2.2,
Gymnasium 1.2.0, SB3 2.7.0, предустановленный Torch 2.11.0 (CPU/CUDA build сохраняется).
Установка без `--upgrade`; новые конфликты блокируют продолжение. Известные системные предупреждения
ipython/jedi и ibis/toolz классифицируются существующей политикой, а не скрываются.
В новой сессии эти ячейки выполняются заново; импорт ранее установленной иной версии требует перезапуска.

In [ ]:
import importlib.metadata

# Read distribution metadata only: do not import NumPy/Pandas before pip.
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None

before_install = {name: installed_version(name) for name in (
    "numpy", "pandas", "torch", "torchvision", "torchaudio", "google-colab",
    "ipython", "jedi", "ibis-framework", "toolz", "numba", "exchange-calendars",
)}
(PREFLIGHT_DIR / "versions-before-install.json").write_text(
    json.dumps(before_install, indent=2), encoding="utf-8")
install_args = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                "--report", str(PREFLIGHT_DIR / "pip-install-report.json"),
                "-r", str(PROJECT_ROOT / "requirements-colab.txt")]
if CHECK_RL_COMPATIBILITY:
    if before_install["torch"] is None:
        raise RuntimeError("В runtime нет PyTorch. Выберите Colab 2026.07; автоматически скачивать/заменять torch не будем.")
    # Keep the exact preinstalled build, including its CUDA/CPU local suffix.
    constraint = PREFLIGHT_DIR / "runtime-constraints.txt"
    constraint.write_text(f"torch=={before_install['torch']}\n", encoding="utf-8")
    install_args += ["-r", str(PROJECT_ROOT / "requirements-rl-check.txt"), "-c", str(constraint)]
install = subprocess.run(install_args, text=True, capture_output=True)
(PREFLIGHT_DIR / "pip-install.log").write_text(install.stdout + "\n" + install.stderr, encoding="utf-8")
if install.returncode != 0:
    print(install.stdout[-4000:])
    print(install.stderr[-4000:])
    raise RuntimeError(f"Установка не прошла: {PREFLIGHT_DIR / 'pip-install.log'}; не обходите конфликт resolver.")
if CHECK_RL_COMPATIBILITY and installed_version("torch") != before_install["torch"]:
    raise RuntimeError("Версия PyTorch изменилась; проверка окружения остановлена")
print("Установка завершена без --upgrade; лог:", PREFLIGHT_DIR / "pip-install.log")

In [ ]:
import importlib
import importlib.metadata
import textwrap

requirements = {}
requirement_files = [PROJECT_ROOT / "requirements-colab.txt"]
if CHECK_RL_COMPATIBILITY:
    requirement_files.append(PROJECT_ROOT / "requirements-rl-check.txt")
for req_file in requirement_files:
    for line in req_file.read_text(encoding="utf-8").splitlines():
        if "==" in line and not line.lstrip().startswith("#"):
            package, version = line.strip().split("==")
            requirements[package] = version
module_names = {"stable-baselines3": "stable_baselines3"}
for package, version in requirements.items():
    installed = importlib.metadata.version(package)
    loaded = getattr(sys.modules.get(module_names.get(package, package)), "__version__", installed)
    if installed != version or loaded != version:
        raise RuntimeError(f"{package}: installed={installed}, loaded={loaded}, expected={version}. "
                           "Перезапустите сеанс Colab и выполните notebook сверху.")

project_env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT), "MPLBACKEND": "Agg"}
smoke_source = textwrap.dedent("""
import importlib, json
names = __MODULES__ + ['src.data', 'src.volatility', 'src.validation', 'src.environment', 'src.portfolio', 'src.environment_data', 'src.portfolio_env', 'src.env_validation', 'src.training_config', 'src.training_data', 'src.regularization', 'src.financial_metrics', 'src.ppo_training', 'src.replay_audit']
modules = {name: importlib.import_module(name) for name in names}
from src.environment import dependency_closure
result = {
    'module_paths': {n: str(m.__file__) for n, m in modules.items()},
    'dependency_closure': dependency_closure(__ROOTS__),
    'future_stack': {'status': 'not_requested'},
}
if __CHECK_RL__:
    import numpy as np
    import torch
    from packaging.version import Version
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_checker import check_env
    if not (Version('2.3') <= Version(torch.__version__) < Version('3.0')):
        raise RuntimeError('PyTorch does not satisfy SB3 2.7.0')
    if Version(torch.__version__).base_version != '2.11.0':
        raise RuntimeError('Expected preinstalled PyTorch 2.11.0 for the selected 2026.07 baseline')
    array = np.array([1.0, 2.0], dtype=np.float32)
    recovered = torch.from_numpy(array).add(1).numpy()
    if not np.array_equal(recovered, np.array([2.0, 3.0], dtype=np.float32)):
        raise RuntimeError('NumPy/PyTorch CPU interoperability check failed')
    result['future_stack'] = {
        'status': 'passed', 'torch': torch.__version__, 'cuda_build': torch.version.cuda,
        'cuda_available': torch.cuda.is_available(), 'numpy_torch_bridge': 'passed',
        'gymnasium': modules['gymnasium'].__version__,
        'stable_baselines3': modules['stable_baselines3'].__version__,
        'PPO_import': 'passed', 'model_created': False, 'training_performed': False,
        'project_environment_created': False,
    }
print(json.dumps(result, indent=2))
""")
smoke_source = (smoke_source.replace("__MODULES__", repr([module_names.get(p, p) for p in requirements]))
                .replace("__ROOTS__", repr(list(requirements) + (["torch"] if CHECK_RL_COMPATIBILITY else [])))
                .replace("__CHECK_RL__", repr(CHECK_RL_COMPATIBILITY)))
smoke = subprocess.run([sys.executable, "-c", smoke_source], cwd=PROJECT_ROOT,
                       env=project_env, capture_output=True, text=True)
(PREFLIGHT_DIR / "imports.log").write_text(smoke.stdout + "\n" + smoke.stderr, encoding="utf-8")
if smoke.returncode != 0:
    print(smoke.stderr[-4000:])
    raise RuntimeError(f"Ошибка import/ABI preflight; см. {PREFLIGHT_DIR / 'imports.log'}")
smoke_report = json.loads(smoke.stdout)

# Equivalent CLI command: python -m pip check (in this Colab kernel's Python).
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
(PREFLIGHT_DIR / "pip-check.txt").write_text(pip_check.stdout + "\n" + pip_check.stderr, encoding="utf-8")
print(pip_check.stdout or pip_check.stderr)
policy_source = (
    "import json, sys; from src.environment import classify_pip_check; "
    "payload=json.load(sys.stdin); print(json.dumps(classify_pip_check(payload['output'], payload['code'])))"
)
policy = subprocess.run([sys.executable, "-c", policy_source], input=json.dumps({
    "output": pip_check.stdout + "\n" + pip_check.stderr, "code": pip_check.returncode,
}), cwd=PROJECT_ROOT, env=project_env, capture_output=True, text=True, check=True)
classification = json.loads(policy.stdout)
environment = {
    "schema": 2, "profile": "env-v2-colab-2026.07", "rl_check_requested": CHECK_RL_COMPATIBILITY,
    "direct_versions": {p: importlib.metadata.version(p) for p in requirements},
    "versions_before_install": before_install,
    "project_imports": "passed", "subprocess_module_paths": smoke_report["module_paths"],
    "dependency_closure": smoke_report["dependency_closure"],
    "future_stack": smoke_report["future_stack"], "pip_check_policy": classification,
    "pip_check_returncode": pip_check.returncode, "drive_read_write": "passed",
    "python": sys.version, "colab_release_tag": os.environ["COLAB_RELEASE_TAG"],
}
(PREFLIGHT_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Фактические версии:", json.dumps(environment["direct_versions"], indent=2))
print("Dependency closure:", smoke_report["dependency_closure"]["status"])
print("Global pip check:", classification["status"])
print("Future stack:", json.dumps(environment["future_stack"], indent=2))
if smoke_report["dependency_closure"]["status"] != "passed" or classification["status"] == "blocked":
    raise RuntimeError(f"Неустранённый или новый конфликт; изучите {PREFLIGHT_DIR / 'environment.json'}")
print("Project dependency preflight passed; известные предупреждения, если есть, сохранены.")

## 4. Конфигурация и сохранение исходников
Все режимы отличаются только `reward_mode`. Общие параметры берутся из JSON; начальные значения
пилотные, не оптимальные. Для отрицательной чувствительности заранее задайте `alpha` в общем JSON,
создайте новый commit и отдельную группу запусков. Не выбирайте alpha по test.
Обработка validation использует замороженную нормализацию train; нормализация reward отключена.

In [ ]:
import shutil
from dataclasses import replace

os.chdir(PROJECT_ROOT)
for name, module in list(sys.modules.items()):
    if name == "src" or name.startswith("src."):
        location = getattr(module, "__file__", None)
        if location and not Path(location).resolve().is_relative_to(PROJECT_ROOT.resolve()):
            raise RuntimeError("src imported from a different checkout; restart session")
sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
from src.data import sha256_file, write_json
from src.validation import pytest_summary, verify_file_map
from src.training_config import load_training_config
from src.training_data import load_development_market
from src.ppo_training import train_one_run, replay_saved_run
from src.replay_audit import audit_saved_policies

COMMON = replace(load_training_config(PROJECT_ROOT / "configs" / f"ppo_{RUN_KIND}.json"), device=DEVICE)
COMMON.validate()
CONFIGS = {mode: replace(COMMON, reward_mode=mode) for mode in REWARD_MODES}
common_payloads = [{k: v for k, v in cfg.to_dict().items() if k != "reward_mode"} for cfg in CONFIGS.values()]
assert all(payload == common_payloads[0] for payload in common_payloads)
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8]
SESSION_DIR = OUTPUT_ROOT / "sessions" / SESSION_ID
SESSION_DIR.mkdir(parents=True, exist_ok=False)
source_paths = sorted(set([
    *PROJECT_ROOT.glob("src/*.py"), *PROJECT_ROOT.glob("tests/test_*.py"),
    *PROJECT_ROOT.glob(".agents/skills/*/SKILL.md"), *PROJECT_ROOT.glob("configs/*.json"),
    *PROJECT_ROOT.glob("docs/*.md"), PROJECT_ROOT / "AGENTS.md",
    PROJECT_ROOT / "requirements-colab.txt", PROJECT_ROOT / "requirements-rl-check.txt",
    PROJECT_ROOT / "notebooks/03_ppo_training.ipynb",
]))
source_hashes = {}
for original in source_paths:
    relative = original.relative_to(PROJECT_ROOT)
    destination = SESSION_DIR / "provenance" / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(original, destination)
    source_hashes[relative.as_posix()] = sha256_file(original)
subprocess.run(["git", "-C", str(REPO_ROOT), "bundle", "create", str(SESSION_DIR / "source.bundle"), "HEAD"], check=True)
for original in PREFLIGHT_DIR.iterdir():
    if original.is_file():
        shutil.copy2(original, SESSION_DIR / original.name)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
(SESSION_DIR / "pip-freeze.txt").write_text(freeze.stdout, encoding="utf-8")
write_json(SESSION_DIR / "common_config.json", COMMON.to_dict())
session = {"stage": 3, "status": "started", "session_id": SESSION_ID, "git_sha": SOURCE_COMMIT,
           "evaluator_git_sha": SOURCE_COMMIT,
           "model_git_sha": SOURCE_COMMIT if EXECUTION_MODE == "train" else MODEL_GIT_SHA,
           "source_sha256": source_hashes, "execution_mode": EXECUTION_MODE, "run_kind": RUN_KIND,
           "tests": {"status": "not_run"}, "runs": {}, "test_used": False, "environment": environment}
write_json(SESSION_DIR / "session.json", session)
print("Session:", SESSION_DIR)
print("Common configuration:", json.dumps(COMMON.to_dict(), indent=2))
print("Real training budget per mode:", COMMON.total_timesteps if EXECUTION_MODE == "train" else 0)

## 5. Автоматические тесты перед новым обучением
`train` выполняет полный pytest до реальных данных. Набор включает маленькие синтетические
обучения и проверки нового аудита; это не реальные исследовательские результаты.
`audit_only` и `reload_only` пропускают pytest, чтобы не вызывать даже синтетическое обучение.
Диагностика окружения выполняется во всех режимах. Старые 97 passed относятся к a768abc;
новые тесты не считать пройденными до фактического выполнения.

In [ ]:
if EXECUTION_MODE == "train":
    test_env = {**project_env, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
    result = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q",
                             f"--junitxml={SESSION_DIR / 'pytest.xml'}"],
                            cwd=PROJECT_ROOT, env=test_env, capture_output=True, text=True)
    (SESSION_DIR / "pytest.log").write_text(result.stdout + "\n" + result.stderr, encoding="utf-8")
    print(result.stdout)
    if result.returncode:
        session.update(status="tests_failed", tests={"status": "failed", "exit_code": result.returncode})
        write_json(SESSION_DIR / "session.json", session)
        raise RuntimeError(f"Tests failed; do not train. See {SESSION_DIR / 'pytest.log'}")
    try:
        summary = pytest_summary(SESSION_DIR / "pytest.xml")
        import xml.etree.ElementTree as ET
        cases = {case.get("name") for case in ET.parse(SESSION_DIR / "pytest.xml").getroot().iter("testcase")}
        required_cases = {"test_tiny_synthetic_training_saves_and_reloads", "test_gymnasium_and_sb3_checkers",
                          "test_training_and_loading_reject_test_data_before_any_run",
                          "test_checkpoint_reload_and_financial_metrics_ignore_reward_mode"}
        if not required_cases.issubset(cases):
            raise RuntimeError("Required Stage 3/checker cases were not collected")
    except Exception:
        session["status"] = "tests_failed"
        write_json(SESSION_DIR / "session.json", session)
        raise
    session["tests"] = {**summary, "exit_code": 0}
else:
    session["tests"] = {"status": f"not_run_{EXECUTION_MODE}", "reason": "No learn calls in recovery session"}
write_json(SESSION_DIR / "session.json", session)
print("Actual test status:", session["tests"])

## 6. Чтение только train/validation
Загрузчик использует разрешённый список файлов и Parquet-фильтр дат: train строго до 2019,
validation строго до 2022. `features_test.parquet` не открывается. Проверка SHA общих ценовых
файлов читает сжатые байты для целостности; расчёты не получают строки 2022+.
Сохранённые признаки заново сверяются с причинной формулой; sigma_ref пересчитывается только на train.
При `reload_only` и `audit_only` обучающая среда не создаётся (train-признаки нужны только для проверки sigma_ref).

In [ ]:
if EXECUTION_MODE == "train" and session["tests"].get("status") != "passed":
    raise RuntimeError("Tests must pass before real data/training")
validation = load_development_market(STAGE1_RUN, "validation")
train = load_development_market(STAGE1_RUN, "train") if EXECUTION_MODE == "train" else None
session["data"] = {"validation": validation.provenance,
                   "train": train.provenance if train is not None else "not_loaded_for_training"}
for label, data in (("train", train), ("validation", validation)):
    if data is not None:
        print(label, len(data.features), str(data.features.index.min().date()),
              str(data.features.index.max().date()), "sigma_ref:", data.sigma_ref)
        assert data.provenance["test_features_read"] is False
        assert data.adjusted_close.index.max().year <= (2018 if label == "train" else 2021)
write_json(SESSION_DIR / "session.json", session)

## 7. Выбранная процедура
`audit_only` использует новый код проверки и старые модели. Проверяются хеши, версии,
исходный training SHA и режим каждой модели. Аудит исполняет все исходные действия,
измеряет отклонение предсказаний на исходной траектории и дважды загружает модель
для оценки в текущем runtime. PPO learn и настройка параметров не выполняются.

`diagnostic_complete` означает только завершение сбора свидетельств. Каждый exact-флаг
остаётся true/false по факту; отсутствие точного совпадения не маскируется допуском.
`reload_only` сохраняет строгий assert и при несовпадении остановится с failed report.

`train` обучает три режима с одинаковыми параметрами и seed; оценивается final checkpoint.
Train/validation начинаются со 100% cash и общего капитала. Промежуточные checkpoints
сохраняются во время rollout, final — после optimizer updates. Обучение в audit не возобновляется.

In [ ]:
run_paths = {}
try:
    if EXECUTION_MODE == "train":
        provenance = {"session_directory": str(SESSION_DIR), "source_sha256": source_hashes,
                      "session_input_sha256": {name: sha256_file(SESSION_DIR / name) for name in
                          ("environment.json", "pip-freeze.txt", "pytest.xml", "pytest.log", "common_config.json")}}
        for mode in REWARD_MODES:
            root = train_one_run(train, validation, CONFIGS[mode], OUTPUT_ROOT / "runs",
                                 git_sha=SOURCE_COMMIT, provenance=provenance)
            run_paths[mode] = root
            session["runs"][mode] = root.name
            write_json(SESSION_DIR / "session.json", session)
    else:
        for mode in REWARD_MODES:
            root = OUTPUT_ROOT / "runs" / RELOAD_RUN_IDS[mode]
            prior = json.loads((root / "metadata.json").read_text(encoding="utf-8"))
            if prior["git_sha"] != MODEL_GIT_SHA or prior["reward_mode"] != mode:
                raise RuntimeError("Saved model does not match MODEL_GIT_SHA/reward mode")
            current_versions = {p: importlib.metadata.version(p) for p in prior["versions"]}
            if current_versions != prior["versions"] or prior["python"] != sys.version:
                raise RuntimeError("Use the original recorded Python and dependency versions")
            run_paths[mode] = root
            session["runs"][mode] = root.name
        if EXECUTION_MODE == "audit_only":
            AUDIT_DIR = audit_saved_policies(validation, run_paths, OUTPUT_ROOT / "replay_audits")
            session["replay_audit_directory"] = str(AUDIT_DIR)
            session["replay_audit_report_sha256"] = sha256_file(AUDIT_DIR / "audit_report.json")
        else:
            for mode, root in run_paths.items():
                replay = replay_saved_run(root, validation, OUTPUT_ROOT / "replays" / SESSION_ID / mode)
                session.setdefault("replays", {})[mode] = str(replay)
        write_json(SESSION_DIR / "session.json", session)
except Exception as error:
    session.update(status="failed", error_type=type(error).__name__, error=str(error))
    write_json(SESSION_DIR / "session.json", session)
    raise
print("Source run paths:", {mode: str(path) for mode, path in run_paths.items()})

## 8. Сводка выбранной процедуры
В `audit_only` выводится отдельная таблица exact-флагов и размеров отклонений.
Это диагностический отчёт, не сравнение финансового превосходства и не статус passed.
В остальных режимах сохраняются графики validation исходных runs и training losses.
Ни один из этих графиков не доказывает оптимальность или статистическое превосходство.

In [ ]:
import pandas as pd
from IPython.display import display

if EXECUTION_MODE == "audit_only":
    audit_report = json.loads((AUDIT_DIR / "audit_report.json").read_text(encoding="utf-8"))
    summary = pd.DataFrame([{
        "mode": mode,
        "saved_actions_finances_exact": checks["saved_action_finances"]["exact"],
        "policy_actions_exact": checks["policy_actions_on_saved_states_exact"],
        "original_closed_loop_exact": checks["closed_loop_original"]["exact"],
        "same_runtime_reload_exact": checks["same_runtime_independent_reload"]["exact"],
        "max_action_difference": checks["max_absolute_action_difference_on_saved_states"],
        "max_nav_difference": checks["max_absolute_nav_difference_closed_loop"],
    } for mode, checks in audit_report["runs"].items()])
    summary.to_csv(SESSION_DIR / "replay_audit_summary.csv", index=False)
    display(summary)
    print("Audit evidence:", AUDIT_DIR)
else:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    metrics_rows = []
    fig, axes = plt.subplots(4, 1, figsize=(12, 14), constrained_layout=True)
    for mode, path in run_paths.items():
        records = pd.read_csv(path / "validation_financials.csv", parse_dates=["date"])
        metrics = json.loads((path / "validation_metrics.json").read_text(encoding="utf-8"))
        metrics_rows.append({"mode": mode, "run_id": path.name, **metrics})
        axes[0].plot(records.date, records.portfolio_value, label=mode)
        axes[1].plot(records.date, records.turnover, label=mode)
        axes[2].plot(records.date, records.lambda_t, label=mode)
        axes[3].plot(records.date, records.regularization_penalty, label=mode)
    for ax, title in zip(axes, ("Validation NAV after commissions", "Executed turnover",
                                "Regularization coefficient", "Reward penalty (not an extra cash cost)")):
        ax.set_title(title); ax.legend(); ax.grid(alpha=.25)
    fig.savefig(SESSION_DIR / "validation_diagnostics.png", dpi=140)
    plt.show()
    metrics_table = pd.DataFrame(metrics_rows)
    metrics_table.to_csv(SESSION_DIR / "validation_summary.csv", index=False)
    display(metrics_table)

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), constrained_layout=True)
    for mode, path in run_paths.items():
        history = json.loads((path / "training_metrics.json").read_text(encoding="utf-8"))["records"]
        for ax, key in zip(axes, ("train/value_loss", "train/policy_gradient_loss")):
            points = [(r["training_step"], r[key]) for r in history if key in r and r[key] is not None]
            if points:
                x, y = zip(*points)
                ax.plot(x, y, label=mode)
    for ax, title in zip(axes, ("PPO value loss", "PPO policy gradient loss")):
        ax.set_title(title); ax.legend(); ax.grid(alpha=.25)
    fig.savefig(SESSION_DIR / "training_diagnostics.png", dpi=140)
    plt.show()

## 9. Итоговая read-back проверка и отчёт
В audit_only сохраняется status=diagnostic_complete и точные флаги проверок;
при расхождении с исходным runtime exact_original_replay остаётся false.
Само завершение диагностики не означает успешный strict replay или полную приёмку этапа.
Пришлите обе папки: новую stage3/sessions/<id> и stage3/replay_audits/<id>.

Для проверки после перезапуска runtime используйте новую сессию Colab:
Runtime → Disconnect and delete runtime. Повторное выполнение ячейки не доказывает перезапуск.
Источник моделей и версия evaluator записываются раздельно. Старые runs остаются неизменяемыми.

In [ ]:
assert set(run_paths) == set(REWARD_MODES), "All three modes must be present"
for mode, root in run_paths.items():
    meta = json.loads((root / "metadata.json").read_text(encoding="utf-8"))
    assert (root / "metadata.sha256").read_text().split() == [sha256_file(root / "metadata.json"), "metadata.json"]
    verify_file_map(root, meta["files_sha256"])
    assert meta["status"] == "complete" and meta["test_used"] is False
    assert meta["training_steps"] == meta["budget"]
    expected_model_sha = SOURCE_COMMIT if EXECUTION_MODE == "train" else MODEL_GIT_SHA
    assert meta["git_sha"] == expected_model_sha and meta["reward_mode"] == mode
    assert meta["config"] == CONFIGS[mode].to_dict(), "Use the source group's registered configuration"
    assert json.loads((root / "reload_check.json").read_text())["status"] == "passed"
    print(mode, "original steps:", meta["training_steps"], "run:", root)
if EXECUTION_MODE == "audit_only":
    audit_report = json.loads((AUDIT_DIR / "audit_report.json").read_text(encoding="utf-8"))
    assert (AUDIT_DIR / "audit_report.sha256").read_text().split() == [sha256_file(AUDIT_DIR / "audit_report.json"), "audit_report.json"]
    verify_file_map(AUDIT_DIR, audit_report["files_sha256"])
    assert audit_report["status"] == "diagnostic_complete" and audit_report["test_used"] is False
    session["original_exact_replay"] = all(c["closed_loop_original"]["exact"] for c in audit_report["runs"].values())
    session["saved_action_finances_exact"] = all(c["saved_action_finances"]["exact"] for c in audit_report["runs"].values())
    session["same_runtime_reload_exact"] = all(c["same_runtime_independent_reload"]["exact"] for c in audit_report["runs"].values())
session.update(status="diagnostic_complete" if EXECUTION_MODE == "audit_only" else "complete",
               completed_at_utc=datetime.now(timezone.utc).isoformat(),
               training_performed_in_this_session=EXECUTION_MODE == "train",
               fresh_session_replay="requires_user_confirmation" if EXECUTION_MODE != "train" else "not_yet_verified")
session["files_sha256"] = {p.relative_to(SESSION_DIR).as_posix(): sha256_file(p)
    for p in sorted(SESSION_DIR.rglob("*")) if p.is_file() and p.name not in {"session.json", "session.sha256"}}
write_json(SESSION_DIR / "session.json", session)
(SESSION_DIR / "session.sha256").write_text(sha256_file(SESSION_DIR / "session.json") + "  session.json\n")
verify_file_map(SESSION_DIR, session["files_sha256"])
print("Execution mode:", EXECUTION_MODE, "status:", session["status"])
print("Evaluator Git SHA:", SOURCE_COMMIT)
print("Model training Git SHA:", session["model_git_sha"])
print("Tests in this session:", session["tests"])
print("Session report:", SESSION_DIR)
if EXECUTION_MODE == "audit_only":
    print("Audit report:", AUDIT_DIR)
    print("Original closed-loop exact:", session["original_exact_replay"])
    print("Saved-action finances exact:", session["saved_action_finances_exact"])
    print("Same-runtime reload exact:", session["same_runtime_reload_exact"])
print("Test data used: False. No hyperparameter search or statistical superiority claim.")